# Decoder Complexity Analysis for Composite DNA
### Parameter count, inference throughput, FLOPs, GPU memory and training-time estimates
### Across all three original error models (EZ17, G15, O17) and four alphabets

This notebook benchmarks the **computational cost** of the four decoders
(Bi-LSTM, Min-Distance, KL-Divergence, Maximum-Likelihood).

**Important — this notebook does not depend on trained weights or datasets.**
Every measurement here (parameter count, throughput, FLOPs, memory, per-epoch
time) is computed on **freshly-initialised models and synthetic random input**.
Complexity is a property of the architecture and the alphabet size, not of the
learned parameters, so the bug-fix in the data-generation / training pipeline
does **not** change any number this notebook produces.

**Updated for consistency with the new (post-bug-fix) pipeline.** The only
substantive change versus the old script:

| Aspect | Old script | This notebook |
|---|---|---|
| `max_coverage` | 25 | **50** (matches `dataset_generator_uniform_composite.py`; used only for the training-time batch-count estimate) |
| KL/ML epsilon | hard-coded `0.01` inside the decoders | passed explicitly as a config value (per-profile `best_eps_klml`) |
| Eta-alphabet vocab | `34` (label `eta0.2`) | `34`, label clarified to `eta0.2_ell5` to match the eta training/generator naming |

All architectural constants (`hidden_dim=128`, `num_layers=2`, etc.) already
match the training scripts and are unchanged.


## Cell 1 — Device configuration

In [1]:
# =============================================================================
# CELL 1: DEVICE CONFIGURATION
# =============================================================================
import os

# Pin GPU if needed (set to "" to let PyTorch choose)
DEVICE_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
   GPU: NVIDIA GeForce RTX 3080


## Cell 2 — Imports

In [2]:
# =============================================================================
# CELL 2: IMPORTS
# =============================================================================
import random
import pickle
import json
import time
import numpy as np
from datetime import datetime
from collections import defaultdict

import torch.nn as nn

print("Imports ready.")

Imports ready.


## Cell 3 — Configuration

`DATASET_CONFIG['max_coverage']` is set to `50` to match
`dataset_generator_uniform_composite.py`. It is used here only to size the
training-time batch-count estimate; it does not affect any other measurement.


In [3]:
# =============================================================================
# CELL 3: CONFIGURATION
# =============================================================================

# ------------------- ERROR-MODEL SPECS (sequence lengths only) -------------------
ERROR_MODELS = {
    "erlich":   {"seq_length": 136, "name": "EZ17"},
    "grass":    {"seq_length": 104, "name": "G15"},
    "organick": {"seq_length": 77,  "name": "O17"},
}

# ------------------- ALPHABET CONFIGURATIONS -------------------
# vocab_size drives the FC-layer width and hence parameter count.
ALPHABET_CONFIGS = {
    "A6":    {"vocab_size": 10, "alphabet_mode": "2mix_only",       "name": r"$\mathcal{A}_6$"},
    "A10":   {"vocab_size": 14, "alphabet_mode": "2mix_3mix",       "name": r"$\mathcal{A}_{10}$"},
    "A11":   {"vocab_size": 15, "alphabet_mode": "2mix_3mix_4mix",  "name": r"$\mathcal{A}_{11}$"},
    "A_eta": {"vocab_size": 34, "alphabet_mode": "eta0.2_ell5",     "name": r"$\mathcal{A}_{0.2}$"},
}

# ------------------- MODEL ARCHITECTURE (must match training) -------------------
MODEL_CONFIG = {
    "input_channels": 4,
    "hidden_dim":     128,
    "num_layers":     2,
    "dropout":        0.2,
    "bidirectional":  True,
}

# ------------------- BENCHMARKING PARAMETERS -------------------
BENCHMARK_CONFIG = {
    "num_warmup_runs":    10,
    "num_benchmark_runs": 100,
    "batch_sizes":        [1, 32, 64, 128, 256, 500],
    "default_batch_size": 500,
}

# ------------------- DATASET PARAMETERS (for training-time estimate only) -------
DATASET_CONFIG = {
    "num_samples":  100000,
    "train_ratio":  0.8,
    "max_coverage": 50,        # <-- matches dataset_generator_uniform_composite.py
}

# ------------------- KL/ML SMOOTHING EPSILON -------------------
# The training pipeline uses a per-profile best epsilon. Complexity / throughput
# are insensitive to its exact value (it only changes a clamp constant), but we
# expose it explicitly rather than hard-coding 0.01 inside the decoders.
EPSILON_KLML = 0.01

# ------------------- TRAINING HYPERPARAMETERS (for the epoch-time estimate) -----
LEARNING_RATE = 1e-3
SEED          = 42

RESULTS_DIR = "./complexity_analysis_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("=" * 64)
print(" COMPLEXITY ANALYSIS CONFIGURATION")
print("=" * 64)
print(f"   Error models : {[v['name'] for v in ERROR_MODELS.values()]}")
print(f"   Alphabets    : {list(ALPHABET_CONFIGS.keys())}")
print(f"   Model        : Bi-LSTM, hidden_dim={MODEL_CONFIG['hidden_dim']}, "
      f"layers={MODEL_CONFIG['num_layers']}")
print(f"   KL/ML epsilon: {EPSILON_KLML:g}")
print(f"   Results dir  : {RESULTS_DIR}")
print("=" * 64)

 COMPLEXITY ANALYSIS CONFIGURATION
   Error models : ['EZ17', 'G15', 'O17']
   Alphabets    : ['A6', 'A10', 'A11', 'A_eta']
   Model        : Bi-LSTM, hidden_dim=128, layers=2
   KL/ML epsilon: 0.01
   Results dir  : ./complexity_analysis_results


## Cell 4 — Seed for reproducibility

In [4]:
# =============================================================================
# CELL 4: SEED FOR REPRODUCIBILITY
# =============================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Random seed set to {SEED}.")

Random seed set to 42.


## Cell 5 — Bi-LSTM model (architecture identical to training)

In [5]:
# =============================================================================
# CELL 5: MODEL DEFINITION  (identical to the training-script architecture)
# =============================================================================
class CompositeDecoderLSTM(nn.Module):
    """Bidirectional LSTM decoder for composite DNA.
    Input (B,4,L) -> Output (B,vocab_size,L)."""

    def __init__(self, config):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0.0,
        )
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])

    def forward(self, x):
        x = x.permute(0, 2, 1)            # (B, 4, L) -> (B, L, 4)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        return logits.permute(0, 2, 1)    # (B, vocab_size, L)


print("Model class ready.")

Model class ready.


## Cell 6 — Baseline decoders

The KL/ML decoders take an explicit `epsilon` argument (defaulting to
`EPSILON_KLML`) instead of a hard-coded literal.


In [6]:
# =============================================================================
# CELL 6: BASELINE DECODERS
# =============================================================================
def min_distance_decoder(obs, ideal_vectors):
    """Minimum Euclidean-distance decoder."""
    dists = torch.sum(
        (obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)


def kl_divergence_decoder(obs, ideal_vectors, epsilon=EPSILON_KLML):
    """KL-divergence decoder with smoothed ideals."""
    ideal_safe = torch.clamp(ideal_vectors.clone(), min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    ce = -(obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmin(ce, dim=-1)


def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=EPSILON_KLML):
    """Maximum-likelihood decoder with smoothed ideals."""
    ideal_safe = torch.clamp(ideal_vectors.clone(), min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    ll = (obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmax(ll, dim=-1)


print(f"Baseline decoders ready (KL/ML epsilon = {EPSILON_KLML:g}).")

Baseline decoders ready (KL/ML epsilon = 0.01).


## Cell 7 — Ideal-vector builders

The four-mix (`A11`) and three-mix (`A10`) ideals match the uniform training
script. The eta-alphabet ideals match `dataset_generator_eta_based.py`
(`eta=0.2`, `ell ∈ {-2,-1,0,1,2}` → 4 pure + 6 pairs × 5 ratios = 34 classes).


In [7]:
# =============================================================================
# CELL 7: IDEAL-VECTOR BUILDERS
# =============================================================================
def build_ideal_vectors_A6():
    return torch.tensor([
        [1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0],
        [0.5, 0.0, 0.0, 0.5], [0.0, 0.5, 0.5, 0.0],
        [0.0, 0.5, 0.0, 0.5], [0.0, 0.0, 0.5, 0.5],
        [0.5, 0.5, 0.0, 0.0], [0.5, 0.0, 0.5, 0.0],
    ], dtype=torch.float32)


def build_ideal_vectors_A10():
    t = 1.0 / 3.0
    base = build_ideal_vectors_A6().tolist()
    base += [[t, t, t, 0.0], [t, t, 0.0, t], [t, 0.0, t, t], [0.0, t, t, t]]
    return torch.tensor(base, dtype=torch.float32)


def build_ideal_vectors_A11():
    base = build_ideal_vectors_A10().tolist()
    base.append([0.25, 0.25, 0.25, 0.25])
    return torch.tensor(base, dtype=torch.float32)


def build_ideal_vectors_eta(eta=0.2, ell_values=(-2, -1, 0, 1, 2)):
    """Eta-based alphabet: 4 pure bases + 6 unordered pairs x len(ell_values)
    ratios. Matches dataset_generator_eta_based.py."""
    vecs = [
        [1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0],
    ]
    pair_indices = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
    for idx1, idx2 in pair_indices:
        for ell in ell_values:
            vec = [0.0, 0.0, 0.0, 0.0]
            vec[idx1] = 0.5 + ell * eta
            vec[idx2] = 0.5 - ell * eta
            vecs.append(vec)
    return torch.tensor(vecs, dtype=torch.float32)


def get_ideal_vectors(alphabet_key):
    builders = {
        "A6":    build_ideal_vectors_A6,
        "A10":   build_ideal_vectors_A10,
        "A11":   build_ideal_vectors_A11,
        "A_eta": build_ideal_vectors_eta,
    }
    if alphabet_key not in builders:
        raise ValueError(f"Unknown alphabet: {alphabet_key}")
    return builders[alphabet_key]()


# Sanity check: each builder's row-count must equal the configured vocab_size.
for _ak, _ac in ALPHABET_CONFIGS.items():
    _n = get_ideal_vectors(_ak).shape[0]
    assert _n == _ac['vocab_size'], (
        f"{_ak}: ideal vectors give {_n} rows but vocab_size={_ac['vocab_size']}")
print("Ideal-vector builders ready (vocab sizes verified).")

Ideal-vector builders ready (vocab sizes verified).


## Cell 8 — Parameter-counting functions

In [8]:
# =============================================================================
# CELL 8: PARAMETER-COUNTING FUNCTIONS
# =============================================================================
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def count_parameters_by_layer(model):
    out = {}
    for name, module in model.named_modules():
        if len(list(module.children())) == 0:
            params = sum(p.numel() for p in module.parameters())
            if params > 0:
                out[name] = params
    return out


def get_lstm_fc_breakdown(model):
    lstm_params = sum(p.numel() for p in model.lstm.parameters())
    fc_params = sum(p.numel() for p in model.fc.parameters())
    return lstm_params, fc_params


def analyze_model_parameters(vocab_size, model_config):
    config = model_config.copy()
    config['vocab_size'] = vocab_size
    model = CompositeDecoderLSTM(config)
    total, trainable = count_parameters(model)
    lstm_params, fc_params = get_lstm_fc_breakdown(model)
    return {
        'total_params':     total,
        'trainable_params': trainable,
        'layer_breakdown':  count_parameters_by_layer(model),
        'lstm_params':      lstm_params,
        'fc_params':        fc_params,
        'model':            model,
    }


print("Parameter-counting helpers ready.")

Parameter-counting helpers ready.


## Cell 9 — Inference-throughput measurement

In [9]:
# =============================================================================
# CELL 9: INFERENCE THROUGHPUT MEASUREMENT
# =============================================================================
def generate_random_input(batch_size, seq_length, device):
    """Random normalised (B,4,L) frequency-matrix input."""
    raw = torch.rand(batch_size, 4, seq_length, device=device)
    return raw / raw.sum(dim=1, keepdim=True)


def measure_lstm_throughput(model, batch_size, seq_length,
                            num_warmup, num_runs, device):
    """Measure Bi-LSTM forward-pass throughput on synthetic input."""
    model.eval()
    model.to(device)
    x = generate_random_input(batch_size, seq_length, device)

    with torch.no_grad():
        for _ in range(num_warmup):
            _ = model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(num_runs):
            _ = model(x)
            if device.type == 'cuda':
                torch.cuda.synchronize()
    total_time = time.perf_counter() - start

    total_symbols = batch_size * seq_length * num_runs
    return {
        'throughput_symbols_per_sec': total_symbols / total_time,
        'latency_ms_per_batch':       (total_time / num_runs) * 1000,
        'total_time_sec':             total_time,
        'num_runs':                   num_runs,
    }


def measure_baseline_throughput(decoder_func, ideal_vectors, batch_size,
                                seq_length, num_warmup, num_runs, device):
    """Measure a baseline decoder's throughput on synthetic input."""
    ideal_vectors = ideal_vectors.to(device)
    raw = torch.rand(batch_size, seq_length, 4, device=device)
    obs = raw / raw.sum(dim=2, keepdim=True)

    for _ in range(num_warmup):
        _ = decoder_func(obs, ideal_vectors)
    if device.type == 'cuda':
        torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(num_runs):
        _ = decoder_func(obs, ideal_vectors)
        if device.type == 'cuda':
            torch.cuda.synchronize()
    total_time = time.perf_counter() - start

    total_symbols = batch_size * seq_length * num_runs
    return {
        'throughput_symbols_per_sec': total_symbols / total_time,
        'latency_ms_per_batch':       (total_time / num_runs) * 1000,
        'total_time_sec':             total_time,
        'num_runs':                   num_runs,
    }


print("Throughput-measurement helpers ready.")

Throughput-measurement helpers ready.


## Cell 10 — Training-time estimation

In [10]:
# =============================================================================
# CELL 10: TRAINING-TIME ESTIMATION
# =============================================================================
def estimate_training_time_per_epoch(model, batch_size, num_samples,
                                     seq_length, device, lr=LEARNING_RATE):
    """Estimate wall-clock time for one training epoch (synthetic batches)."""
    model.train()
    model.to(device)
    num_batches = num_samples // batch_size

    x = generate_random_input(batch_size, seq_length, device)
    labels = torch.randint(0, model.fc.out_features,
                           (batch_size, seq_length), device=device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for _ in range(5):  # warmup
        optimizer.zero_grad()
        loss = criterion(model(x), labels)
        loss.backward()
        optimizer.step()
    if device.type == 'cuda':
        torch.cuda.synchronize()

    num_measure_batches = min(20, num_batches)
    start = time.perf_counter()
    for _ in range(num_measure_batches):
        optimizer.zero_grad()
        loss = criterion(model(x), labels)
        loss.backward()
        optimizer.step()
        if device.type == 'cuda':
            torch.cuda.synchronize()
    time_per_batch = (time.perf_counter() - start) / num_measure_batches
    estimated_epoch_time = time_per_batch * num_batches

    return {
        'time_per_batch_sec':        time_per_batch,
        'estimated_epoch_time_sec':  estimated_epoch_time,
        'estimated_epoch_time_min':  estimated_epoch_time / 60,
        'num_batches_per_epoch':     num_batches,
    }


print("Training-time estimation helper ready.")

Training-time estimation helper ready.


## Cell 11 — Time-complexity & FLOPs analysis

In [11]:
# =============================================================================
# CELL 11: TIME-COMPLEXITY & FLOPs ANALYSIS
# =============================================================================
def get_time_complexity_string(decoder_type, model_config=None):
    """Big-O time complexity per sequence position."""
    if decoder_type in ('min_distance', 'kl', 'ml'):
        return r"$\mathcal{O}(|\mathcal{A}|)$"
    if decoder_type == 'lstm':
        return r"$\mathcal{O}(d_h^2)$"
    return "Unknown"


def estimate_lstm_flops_per_position(input_size, hidden_size, num_layers,
                                     bidirectional, vocab_size):
    """Approximate FLOPs per sequence position for the Bi-LSTM + FC head.

    LSTM cell: ~8 * hidden * (input + hidden) MACs per layer per direction.
    FC head  : 2 * (hidden*directions) * vocab_size.
    """
    directions = 2 if bidirectional else 1

    flops_layer1 = 8 * hidden_size * (input_size + hidden_size) * directions

    input_size_subsequent = hidden_size * directions
    flops_per_subseq = 8 * hidden_size * (input_size_subsequent + hidden_size) * directions
    flops_other = flops_per_subseq * (num_layers - 1)

    fc_input = hidden_size * directions
    flops_fc = 2 * fc_input * vocab_size

    return flops_layer1 + flops_other + flops_fc


def estimate_baseline_flops_per_position(vocab_size, decoder_type):
    """Approximate FLOPs per position for a baseline decoder:
    ~12 ops per symbol (4 sub/mul + 4 sq/log + 3 add + 1 compare)."""
    return 12 * vocab_size


print("Complexity / FLOPs helpers ready.")

Complexity / FLOPs helpers ready.


## Cell 12 — GPU-memory estimation

In [12]:
# =============================================================================
# CELL 12: GPU-MEMORY ESTIMATION
# =============================================================================
def estimate_model_memory(model, batch_size, seq_length, device):
    """Estimate GPU memory during a single inference pass."""
    model.eval()
    model.to(device)
    if device.type != 'cuda':
        return {'error': 'Memory measurement requires CUDA'}

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    x = generate_random_input(batch_size, seq_length, device)
    torch.cuda.synchronize()
    memory_before = torch.cuda.memory_allocated()

    with torch.no_grad():
        _ = model(x)
        torch.cuda.synchronize()

    memory_after = torch.cuda.max_memory_allocated()
    model_memory = sum(p.numel() * p.element_size() for p in model.parameters())

    return {
        'inference_memory_mb': (memory_after - memory_before) / (1024 * 1024),
        'peak_memory_mb':      memory_after / (1024 * 1024),
        'model_memory_mb':     model_memory / (1024 * 1024),
    }


print("Memory-estimation helper ready.")

Memory-estimation helper ready.


## Cell 13 — Analysis 1: parameter counting

In [13]:
# =============================================================================
# CELL 13: ANALYSIS 1 -- PARAMETER COUNTING
# =============================================================================
print("=" * 70)
print(" ANALYSIS 1: PARAMETER COUNTING")
print("=" * 70)
print("(Parameters depend only on vocab_size, not on the error model.)")

param_results = {}
for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    analysis = analyze_model_parameters(alphabet_config['vocab_size'], MODEL_CONFIG)
    param_results[alphabet_key] = analysis
    print(f"\n{alphabet_config['name']} (vocab_size={alphabet_config['vocab_size']}):")
    print(f"   Total parameters: {analysis['total_params']:,}")
    print(f"   LSTM parameters : {analysis['lstm_params']:,}")
    print(f"   FC parameters   : {analysis['fc_params']:,}")

print(f"\n{'='*70}")
print(" PARAMETER COUNT SUMMARY")
print(f"{'='*70}")
print(f"{'Decoder':<30} {'Total Params':>15} {'LSTM':>12} {'FC':>12}")
print(f"{'-'*69}")
for name in ('Min. Distance', 'KL Divergence', 'Max. Likelihood'):
    print(f"{name:<30} {'0':>15} {'N/A':>12} {'N/A':>12}")
for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    name = f"Bi-LSTM ({alphabet_config['name']})"
    r = param_results[alphabet_key]
    print(f"{name:<30} {r['total_params']:>15,} "
          f"{r['lstm_params']:>12,} {r['fc_params']:>12,}")

 ANALYSIS 1: PARAMETER COUNTING
(Parameters depend only on vocab_size, not on the error model.)

$\mathcal{A}_6$ (vocab_size=10):
   Total parameters: 535,050
   LSTM parameters : 532,480
   FC parameters   : 2,570

$\mathcal{A}_{10}$ (vocab_size=14):
   Total parameters: 536,078
   LSTM parameters : 532,480
   FC parameters   : 3,598

$\mathcal{A}_{11}$ (vocab_size=15):
   Total parameters: 536,335
   LSTM parameters : 532,480
   FC parameters   : 3,855

$\mathcal{A}_{0.2}$ (vocab_size=34):
   Total parameters: 541,218
   LSTM parameters : 532,480
   FC parameters   : 8,738

 PARAMETER COUNT SUMMARY
Decoder                           Total Params         LSTM           FC
---------------------------------------------------------------------
Min. Distance                                0          N/A          N/A
KL Divergence                                0          N/A          N/A
Max. Likelihood                              0          N/A          N/A
Bi-LSTM ($\mathcal{A}_6$)     

## Cell 14 — Analysis 2: inference throughput (all error models)

In [14]:
# =============================================================================
# CELL 14: ANALYSIS 2 -- INFERENCE THROUGHPUT (all error models)
# =============================================================================
print("=" * 70)
print(" ANALYSIS 2: INFERENCE THROUGHPUT")
print("=" * 70)

throughput_results = defaultdict(lambda: defaultdict(dict))
batch_size = BENCHMARK_CONFIG['default_batch_size']
num_warmup = BENCHMARK_CONFIG['num_warmup_runs']
num_runs   = BENCHMARK_CONFIG['num_benchmark_runs']

for error_key, error_config in ERROR_MODELS.items():
    seq_length = error_config['seq_length']
    print(f"\n{'='*50}")
    print(f" Error model: {error_config['name']} (seq_length={seq_length})")
    print(f"{'='*50}")

    for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
        vocab_size = alphabet_config['vocab_size']
        print(f"\n   {alphabet_config['name']} (vocab_size={vocab_size}):")
        ideal_vectors = get_ideal_vectors(alphabet_key)

        # Bi-LSTM
        config = MODEL_CONFIG.copy()
        config['vocab_size'] = vocab_size
        model = CompositeDecoderLSTM(config)
        lstm_result = measure_lstm_throughput(
            model, batch_size, seq_length, num_warmup, num_runs, device)
        throughput_results[error_key][alphabet_key]['Bi-LSTM'] = lstm_result
        print(f"      Bi-LSTM       : {lstm_result['throughput_symbols_per_sec']:,.0f} symbols/sec")

        # Baselines
        for label, key, func in [
            ('Min Distance',   'Min_Distance', min_distance_decoder),
            ('KL Divergence',  'KL',           kl_divergence_decoder),
            ('Max Likelihood', 'ML',           maximum_likelihood_decoder),
        ]:
            res = measure_baseline_throughput(
                func, ideal_vectors, batch_size, seq_length,
                num_warmup, num_runs, device)
            throughput_results[error_key][alphabet_key][key] = res
            print(f"      {label:<14}: {res['throughput_symbols_per_sec']:,.0f} symbols/sec")

 ANALYSIS 2: INFERENCE THROUGHPUT

 Error model: EZ17 (seq_length=136)

   $\mathcal{A}_6$ (vocab_size=10):
      Bi-LSTM       : 5,630,176 symbols/sec
      Min Distance  : 459,306,708 symbols/sec
      KL Divergence : 276,314,038 symbols/sec
      Max Likelihood: 325,719,717 symbols/sec

   $\mathcal{A}_{10}$ (vocab_size=14):
      Bi-LSTM       : 5,663,771 symbols/sec
      Min Distance  : 436,609,875 symbols/sec
      KL Divergence : 383,910,507 symbols/sec
      Max Likelihood: 402,735,278 symbols/sec

   $\mathcal{A}_{11}$ (vocab_size=15):
      Bi-LSTM       : 5,666,877 symbols/sec
      Min Distance  : 414,501,657 symbols/sec
      KL Divergence : 371,065,918 symbols/sec
      Max Likelihood: 394,205,824 symbols/sec

   $\mathcal{A}_{0.2}$ (vocab_size=34):
      Bi-LSTM       : 5,668,274 symbols/sec
      Min Distance  : 190,987,121 symbols/sec
      KL Divergence : 204,191,953 symbols/sec
      Max Likelihood: 222,448,322 symbols/sec

 Error model: G15 (seq_length=104)

   $\m

## Cell 15 — Analysis 3: training-time estimation (all error models)

In [15]:
# =============================================================================
# CELL 15: ANALYSIS 3 -- TRAINING-TIME ESTIMATION (all error models)
# =============================================================================
print("=" * 70)
print(" ANALYSIS 3: TRAINING-TIME ESTIMATION")
print("=" * 70)

training_results = defaultdict(dict)
num_samples = int(DATASET_CONFIG['num_samples'] * DATASET_CONFIG['train_ratio'])

for error_key, error_config in ERROR_MODELS.items():
    seq_length = error_config['seq_length']
    print(f"\n Error model: {error_config['name']} (seq_length={seq_length})")
    for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
        config = MODEL_CONFIG.copy()
        config['vocab_size'] = alphabet_config['vocab_size']
        model = CompositeDecoderLSTM(config)
        train_result = estimate_training_time_per_epoch(
            model, batch_size, num_samples, seq_length, device)
        training_results[error_key][alphabet_key] = train_result
        print(f"   {alphabet_config['name']}: "
              f"{train_result['estimated_epoch_time_min']:.2f} min/epoch")

 ANALYSIS 3: TRAINING-TIME ESTIMATION

 Error model: EZ17 (seq_length=136)
   $\mathcal{A}_6$: 0.08 min/epoch
   $\mathcal{A}_{10}$: 0.08 min/epoch
   $\mathcal{A}_{11}$: 0.08 min/epoch
   $\mathcal{A}_{0.2}$: 0.08 min/epoch

 Error model: G15 (seq_length=104)
   $\mathcal{A}_6$: 0.06 min/epoch
   $\mathcal{A}_{10}$: 0.06 min/epoch
   $\mathcal{A}_{11}$: 0.06 min/epoch
   $\mathcal{A}_{0.2}$: 0.06 min/epoch

 Error model: O17 (seq_length=77)
   $\mathcal{A}_6$: 0.05 min/epoch
   $\mathcal{A}_{10}$: 0.05 min/epoch
   $\mathcal{A}_{11}$: 0.05 min/epoch
   $\mathcal{A}_{0.2}$: 0.05 min/epoch


## Cell 16 — Analysis 4: FLOPs per position

In [16]:
# =============================================================================
# CELL 16: ANALYSIS 4 -- FLOPs PER POSITION
# =============================================================================
print("=" * 70)
print(" ANALYSIS 4: FLOPs PER POSITION")
print("=" * 70)
print("(FLOPs per position are independent of sequence length.)")

flops_results = {}
for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    vocab_size = alphabet_config['vocab_size']
    lstm_flops = estimate_lstm_flops_per_position(
        input_size=MODEL_CONFIG['input_channels'],
        hidden_size=MODEL_CONFIG['hidden_dim'],
        num_layers=MODEL_CONFIG['num_layers'],
        bidirectional=MODEL_CONFIG['bidirectional'],
        vocab_size=vocab_size)
    mindist_flops = estimate_baseline_flops_per_position(vocab_size, 'min_distance')
    kl_flops = estimate_baseline_flops_per_position(vocab_size, 'kl')

    flops_results[alphabet_key] = {
        'Bi-LSTM': lstm_flops, 'Min_Distance': mindist_flops,
        'KL': kl_flops, 'ML': kl_flops,
    }
    print(f"\n{alphabet_config['name']} (vocab_size={vocab_size}):")
    print(f"   Bi-LSTM      : {lstm_flops:,} FLOPs/position ({lstm_flops/1000:.1f}K)")
    print(f"   Min Distance : {mindist_flops:,} FLOPs/position")
    print(f"   KL/ML        : {kl_flops:,} FLOPs/position")

 ANALYSIS 4: FLOPs PER POSITION
(FLOPs per position are independent of sequence length.)

$\mathcal{A}_6$ (vocab_size=10):
   Bi-LSTM      : 1,061,888 FLOPs/position (1061.9K)
   Min Distance : 120 FLOPs/position
   KL/ML        : 120 FLOPs/position

$\mathcal{A}_{10}$ (vocab_size=14):
   Bi-LSTM      : 1,063,936 FLOPs/position (1063.9K)
   Min Distance : 168 FLOPs/position
   KL/ML        : 168 FLOPs/position

$\mathcal{A}_{11}$ (vocab_size=15):
   Bi-LSTM      : 1,064,448 FLOPs/position (1064.4K)
   Min Distance : 180 FLOPs/position
   KL/ML        : 180 FLOPs/position

$\mathcal{A}_{0.2}$ (vocab_size=34):
   Bi-LSTM      : 1,074,176 FLOPs/position (1074.2K)
   Min Distance : 408 FLOPs/position
   KL/ML        : 408 FLOPs/position


## Cell 17 — Analysis 5: GPU-memory usage

In [17]:
# =============================================================================
# CELL 17: ANALYSIS 5 -- GPU-MEMORY USAGE
# =============================================================================
print("=" * 70)
print(" ANALYSIS 5: GPU-MEMORY USAGE")
print("=" * 70)

memory_results = defaultdict(dict)
if device.type == 'cuda':
    # EZ17 has the longest sequences -> worst-case memory reference.
    ref_error = 'erlich'
    seq_length = ERROR_MODELS[ref_error]['seq_length']
    print(f"(Using {ERROR_MODELS[ref_error]['name']} seq_length={seq_length} "
          f"as reference.)")

    for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
        config = MODEL_CONFIG.copy()
        config['vocab_size'] = alphabet_config['vocab_size']
        model = CompositeDecoderLSTM(config)
        mem = estimate_model_memory(model, batch_size, seq_length, device)
        memory_results[alphabet_key] = mem
        print(f"\n{alphabet_config['name']} (batch_size={batch_size}):")
        print(f"   Model memory     : {mem['model_memory_mb']:.2f} MB")
        print(f"   Inference memory : {mem['inference_memory_mb']:.2f} MB")
        print(f"   Peak memory      : {mem['peak_memory_mb']:.2f} MB")
else:
    print("GPU not available -- skipping memory measurement.")

 ANALYSIS 5: GPU-MEMORY USAGE
(Using EZ17 seq_length=136 as reference.)

$\mathcal{A}_6$ (batch_size=500):
   Model memory     : 2.04 MB
   Inference memory : 1039.94 MB
   Peak memory      : 1061.86 MB

$\mathcal{A}_{10}$ (batch_size=500):
   Model memory     : 2.04 MB
   Inference memory : 1039.94 MB
   Peak memory      : 1061.86 MB

$\mathcal{A}_{11}$ (batch_size=500):
   Model memory     : 2.05 MB
   Inference memory : 1039.94 MB
   Peak memory      : 1061.86 MB

$\mathcal{A}_{0.2}$ (batch_size=500):
   Model memory     : 2.06 MB
   Inference memory : 1039.94 MB
   Peak memory      : 1061.88 MB


## Cell 18 — LaTeX table for the paper

In [18]:
# =============================================================================
# CELL 18: GENERATE LATEX TABLE
# =============================================================================
print("=" * 70)
print(" LATEX TABLE FOR PAPER")
print("=" * 70)

latex_table = r"""
\begin{table}[t]
\caption{Computational complexity comparison of decoders}
\centering
\renewcommand{\arraystretch}{1.2}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{l|c|c|c}
\toprule
\textbf{Decoder} & \textbf{Parameters} & \textbf{Time Complexity} & \textbf{Training} \\
\midrule
"""

latex_table += r"Min.\ Distance & 0 & $\mathcal{O}(|\mathcal{A}|)$ & None \\" + "\n"
latex_table += r"KL Divergence & 0 & $\mathcal{O}(|\mathcal{A}|)$ & None \\" + "\n"
latex_table += r"Max.\ Likelihood & 0 & $\mathcal{O}(|\mathcal{A}|)$ & None \\" + "\n"
latex_table += r"\midrule" + "\n"

for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    name = (alphabet_config['name']
            .replace('$\\mathcal{', '').replace('}$', '').replace('\\', ''))
    params = param_results[alphabet_key]['total_params']
    if params >= 1e6:
        params_str = f"$\\sim${params/1e6:.2f}M"
    else:
        params_str = f"$\\sim${params/1000:.0f}K"
    latex_table += (f"Bi-LSTM ($\\mathcal{{{name}}}$) & {params_str} "
                    f"& $\\mathcal{{O}}(d_h^2)$ & Required \\\\\n")

latex_table += r"""\bottomrule
\end{tabular}
\label{Table:Complexity}
\end{table}
"""

print(latex_table)

latex_path = os.path.join(RESULTS_DIR, "complexity_table.tex")
with open(latex_path, 'w') as f:
    f.write(latex_table)
print(f"LaTeX table saved: {latex_path}")

 LATEX TABLE FOR PAPER

\begin{table}[t]
\caption{Computational complexity comparison of decoders}
\centering
\renewcommand{\arraystretch}{1.2}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{l|c|c|c}
\toprule
\textbf{Decoder} & \textbf{Parameters} & \textbf{Time Complexity} & \textbf{Training} \\
\midrule
Min.\ Distance & 0 & $\mathcal{O}(|\mathcal{A}|)$ & None \\
KL Divergence & 0 & $\mathcal{O}(|\mathcal{A}|)$ & None \\
Max.\ Likelihood & 0 & $\mathcal{O}(|\mathcal{A}|)$ & None \\
\midrule
Bi-LSTM ($\mathcal{A}_6$}$) & $\sim$535K & $\mathcal{O}(d_h^2)$ & Required \\
Bi-LSTM ($\mathcal{A}_{10}$) & $\sim$536K & $\mathcal{O}(d_h^2)$ & Required \\
Bi-LSTM ($\mathcal{A}_{11}$) & $\sim$536K & $\mathcal{O}(d_h^2)$ & Required \\
Bi-LSTM ($\mathcal{A}_{0.2}$) & $\sim$541K & $\mathcal{O}(d_h^2)$ & Required \\
\bottomrule
\end{tabular}
\label{Table:Complexity}
\end{table}

LaTeX table saved: ./complexity_analysis_results/complexity_table.tex


## Cell 19 — Detailed summary tables

In [19]:
# =============================================================================
# CELL 19: DETAILED SUMMARY TABLES
# =============================================================================
print("=" * 70)
print(" DETAILED SUMMARY TABLES")
print("=" * 70)

# --- Throughput summary (averaged across error models) ---
print("\n--- THROUGHPUT SUMMARY (symbols/sec, averaged over error models) ---")
print(f"{'Alphabet':<12} {'Bi-LSTM':>15} {'Min Dist':>15} {'KL':>15} {'ML':>15}")
print(f"{'-'*72}")
for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    lstm_avg = np.mean([throughput_results[ek][alphabet_key]['Bi-LSTM']['throughput_symbols_per_sec']
                        for ek in ERROR_MODELS])
    mindist_avg = np.mean([throughput_results[ek][alphabet_key]['Min_Distance']['throughput_symbols_per_sec']
                           for ek in ERROR_MODELS])
    kl_avg = np.mean([throughput_results[ek][alphabet_key]['KL']['throughput_symbols_per_sec']
                      for ek in ERROR_MODELS])
    ml_avg = np.mean([throughput_results[ek][alphabet_key]['ML']['throughput_symbols_per_sec']
                      for ek in ERROR_MODELS])
    name = (alphabet_config['name']
            .replace('$\\mathcal{', '').replace('}$', '').replace('\\', ''))
    print(f"{name:<12} {lstm_avg:>15,.0f} {mindist_avg:>15,.0f} "
          f"{kl_avg:>15,.0f} {ml_avg:>15,.0f}")

# --- Training-time summary ---
print("\n--- TRAINING TIME SUMMARY (minutes/epoch) ---")
print(f"{'Alphabet':<12}", end="")
for ec in ERROR_MODELS.values():
    print(f" {ec['name']:>10}", end="")
print()
print(f"{'-'*42}")
for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    name = (alphabet_config['name']
            .replace('$\\mathcal{', '').replace('}$', '').replace('\\', ''))
    print(f"{name:<12}", end="")
    for ek in ERROR_MODELS:
        t = training_results[ek][alphabet_key]['estimated_epoch_time_min']
        print(f" {t:>10.2f}", end="")
    print()

 DETAILED SUMMARY TABLES

--- THROUGHPUT SUMMARY (symbols/sec, averaged over error models) ---
Alphabet             Bi-LSTM        Min Dist              KL              ML
------------------------------------------------------------------------
A}_6$              5,437,846     472,666,435     317,748,299     344,879,542
A}_{10             5,443,800     407,340,548     333,920,305     350,100,114
A}_{11             5,439,103     388,079,675     323,616,842     341,046,963
A}_{0.2            5,425,771     184,711,062     187,738,804     203,001,986

--- TRAINING TIME SUMMARY (minutes/epoch) ---
Alphabet           EZ17        G15        O17
------------------------------------------
A}_6$              0.08       0.06       0.05
A}_{10             0.08       0.06       0.05
A}_{11             0.08       0.06       0.05
A}_{0.2            0.08       0.06       0.05


## Cell 20 — Save all results

In [20]:
# =============================================================================
# CELL 20: SAVE ALL RESULTS
# =============================================================================
print("=" * 70)
print(" SAVING RESULTS")
print("=" * 70)

def convert_to_serializable(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [convert_to_serializable(v) for v in obj]
    return obj


results_summary = {
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'device':    str(device),
    'gpu_name':  torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',
    'config': {
        'model':         MODEL_CONFIG,
        'benchmark':     BENCHMARK_CONFIG,
        'dataset':       DATASET_CONFIG,
        'epsilon_klml':  EPSILON_KLML,
        'error_models':  ERROR_MODELS,
        'alphabets':     {k: {kk: vv for kk, vv in v.items() if kk != 'model'}
                          for k, v in ALPHABET_CONFIGS.items()},
    },
    'parameter_counts': {
        ak: {
            'total':     param_results[ak]['total_params'],
            'trainable': param_results[ak]['trainable_params'],
            'lstm':      param_results[ak]['lstm_params'],
            'fc':        param_results[ak]['fc_params'],
        } for ak in ALPHABET_CONFIGS
    },
    'throughput':         convert_to_serializable(dict(throughput_results)),
    'training_time':      convert_to_serializable(dict(training_results)),
    'flops_per_position': flops_results,
    'memory': (convert_to_serializable(dict(memory_results))
               if memory_results else 'N/A'),
}

results_path = os.path.join(RESULTS_DIR, "complexity_analysis_all_models.json")
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=4)
print(f"Results saved: {results_path}")

 SAVING RESULTS
Results saved: ./complexity_analysis_results/complexity_analysis_all_models.json


## Cell 21 — Final paper-ready summary

In [21]:
# =============================================================================
# CELL 21: FINAL PAPER-READY SUMMARY
# =============================================================================
print("=" * 70)
print(" FINAL PAPER-READY SUMMARY")
print("=" * 70)

print("\n=== VALUES FOR TABLE IN PAPER ===\n")
print("Baseline decoders (Min. Distance, KL Divergence, Max. Likelihood):")
print("  - Parameters     : 0")
print("  - Time complexity: O(|A|) per position")
print("  - Training       : None")
print("  - Note           : require only ideal frequency vectors, no learned params")

print("\nBi-LSTM decoders:")
for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    r = param_results[alphabet_key]
    avg_tp = np.mean([throughput_results[ek][alphabet_key]['Bi-LSTM']['throughput_symbols_per_sec']
                      for ek in ERROR_MODELS])
    avg_train = np.mean([training_results[ek][alphabet_key]['estimated_epoch_time_min']
                         for ek in ERROR_MODELS])
    print(f"\n  {alphabet_config['name']} (vocab_size={alphabet_config['vocab_size']}):")
    print(f"    - Total parameters : {r['total_params']:,} (~{r['total_params']/1000:.0f}K)")
    print(f"      - LSTM: {r['lstm_params']:,}, FC: {r['fc_params']:,}")
    print(f"    - Time complexity  : O(d_h^2) per position, d_h={MODEL_CONFIG['hidden_dim']}")
    print(f"    - Avg throughput   : {avg_tp:,.0f} symbols/sec ({avg_tp/1e6:.2f}M)")
    print(f"    - Avg training time: ~{avg_train:.1f} min/epoch")
    print(f"    - Training         : Required")

print("\n" + "=" * 70)
print(" COMPLEXITY ANALYSIS COMPLETE")
print(f"   Results saved to: {RESULTS_DIR}")
print("=" * 70)

 FINAL PAPER-READY SUMMARY

=== VALUES FOR TABLE IN PAPER ===

Baseline decoders (Min. Distance, KL Divergence, Max. Likelihood):
  - Parameters     : 0
  - Time complexity: O(|A|) per position
  - Training       : None
  - Note           : require only ideal frequency vectors, no learned params

Bi-LSTM decoders:

  $\mathcal{A}_6$ (vocab_size=10):
    - Total parameters : 535,050 (~535K)
      - LSTM: 532,480, FC: 2,570
    - Time complexity  : O(d_h^2) per position, d_h=128
    - Avg throughput   : 5,437,846 symbols/sec (5.44M)
    - Avg training time: ~0.1 min/epoch
    - Training         : Required

  $\mathcal{A}_{10}$ (vocab_size=14):
    - Total parameters : 536,078 (~536K)
      - LSTM: 532,480, FC: 3,598
    - Time complexity  : O(d_h^2) per position, d_h=128
    - Avg throughput   : 5,443,800 symbols/sec (5.44M)
    - Avg training time: ~0.1 min/epoch
    - Training         : Required

  $\mathcal{A}_{11}$ (vocab_size=15):
    - Total parameters : 536,335 (~536K)
      - LST